# 05 · Adversarial robustness

**RQ3.** How much does an evasion attack degrade each detector, and does the graph representation make evasion harder?

## Attacks evaluated

| Domain | Attack | What changes | Intensities | Clean point |
|---|---|---|---|---|
| NIDS | **Packet size** | feature [2] *Fwd Packet Length Mean* × scale (padding / fragmentation) | 0.1×, 0.5×, 0.8×, 1×, 2×, 5× | 1.0× |
| NIDS | **Inter-arrival time** | feature [4] *Flow IAT Mean* × scale (throttling / jitter) | 1×, 2×, 5×, 10× | 1.0× |
| PE | **GAN evasion** | malware node features moved toward a GAN sample: x' = x + scale·(G(z) − x), clipped to the real range | 0, 0.25, 0.5, 0.75, 1.0 | 0.0 |

Notes on the setup (`src/evaluation/nids_adversarial.py`, `pe_adversarial.py`):
* NIDS: all windows are perturbed (benign too), as a network-level change would be. The scale is applied to the
  **raw** feature of every flow (`f' = scale · f`, then re-`log1p`), and the baselines receive the mean pooling of the
  *same* perturbed graphs, so GNN and baselines face exactly the same attack.
* PE: **only malware** is perturbed, the attacker's goal being to get a malicious file classified as benign.
  Therefore **recall** (detection rate on malware) is the main metric, and FPR should stay constant.
* Attacks are black-box and feature-space: they do not check that the modified traffic/binary is still functional, so they
  are an **upper bound** on what a real attacker could achieve with these manipulations.

In [ ]:
import sys
import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # torch.jit / pandas deprecations
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm without ipywidgets
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils.config import CONFIG

EXP = project_root / CONFIG["experiments_path"]
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})
GNN_NAME = "GNN (MFGraph)"


def latest_run(base: Path, filename: str):
    """Most recent timestamped run folder under `base` that contains `filename`."""
    runs = sorted(p.parent for p in base.glob(f"*/{filename}")) if base.exists() else []
    return runs[-1] if runs else None


def missing(path, task):
    print(f"[Missing] {path}\nRun first:  python orchestrator.py -t {task}")

MARKERS = ["o", "s", "^", "D", "v", "<", ">", "p", "*"]


def load(path, task):
    if not path.exists():
        missing(path, task)
        return None
    df = pd.read_csv(path)
    df["scale_num"] = df["Scale"].astype(str).str.rstrip("x").astype(float)
    return df.sort_values("scale_num")


def plot_attack(df, title, xlabel, metrics=("F1", "Recall", "FPR")):
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4.8))
    for ax, metric in zip(axes, metrics):
        for i, (model, g) in enumerate(df.groupby("Model")):
            gnn = model == GNN_NAME
            ax.plot(g["scale_num"].astype(str) + "×", g[metric], marker=MARKERS[i % len(MARKERS)],
                    lw=3 if gnn else 1.3, zorder=3 if gnn else 2, label=model)
        ax.set_title(f"{metric}"); ax.set_xlabel(xlabel); ax.set_ylim(-0.02, 1.05)
    axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.suptitle(title, fontweight="bold", y=1.02); plt.tight_layout(); plt.show()


def robustness_table(df, clean_scale, metric="Recall"):
    """Clean value, worst value under attack, and the relative drop."""
    clean = df[df.scale_num == clean_scale].set_index("Model")[metric]
    attacked = df[df.scale_num != clean_scale]
    worst = attacked.groupby("Model")[metric].min()
    worst_at = attacked.loc[attacked.groupby("Model")[metric].idxmin()].set_index("Model")["Scale"]
    t = pd.DataFrame({f"clean {metric}": clean, f"worst {metric}": worst, "worst at": worst_at})
    t["relative drop %"] = 100 * (t[f"clean {metric}"] - t[f"worst {metric}"]) / t[f"clean {metric}"].replace(0, np.nan)
    return t.sort_values("relative drop %")

## 1. NIDS — packet size manipulation

In [ ]:
ps = load(EXP / "nids_adversarial" / "Packet_Size_multimetric.csv", "eval_nids_adversarial")
if ps is not None:
    plot_attack(ps, "NIDS under packet-size manipulation", "Scale on Fwd Packet Length Mean")
    display(robustness_table(ps, 1.0).round(3))

Scales < 1 simulate fragmentation, scales > 1 padding. A model whose recall collapses on one side only has learned a
**threshold on packet size** — exactly the kind of shortcut an attacker can exploit cheaply.

## 2. NIDS — inter-arrival time manipulation

In [ ]:
iat = load(EXP / "nids_adversarial" / "Inter-Arrival_Time_multimetric.csv", "eval_nids_adversarial")
if iat is not None:
    plot_attack(iat, "NIDS under inter-arrival-time manipulation", "Scale on Flow IAT Mean")
    display(robustness_table(iat, 1.0).round(3))

Slowing traffic down (larger IAT) is the classic "low and slow" evasion. It is also costly for the attacker (DoS becomes
less effective), so robustness at 2–5× matters more than at 10×.

## 3. PE — GAN-driven feature evasion

In [ ]:
pe = load(EXP / "pe_adversarial" / "GAN_Evasion_multimetric.csv", "eval_pe_adversarial")
if pe is not None:
    plot_attack(pe, "PE detectors under GAN evasion (malware only perturbed)", "Perturbation intensity")
    display(robustness_table(pe, 0.0).round(3))

Because only malware is perturbed, FPR should stay flat; if it moves, the model's threshold is being dragged by the
batch statistics or the perturbation is leaking into benign samples, which would need investigating.

## 4. Robustness summary across attacks

Relative recall drop (clean → worst intensity) for every model and attack. Lower is better.

In [ ]:
summary = {}
for name, df, clean_scale in [("Packet size", ps, 1.0), ("Inter-arrival", iat, 1.0), ("PE GAN", pe, 0.0)]:
    if df is not None:
        summary[name] = robustness_table(df, clean_scale)["relative drop %"]
if summary:
    s = pd.DataFrame(summary)
    plt.figure(figsize=(8, 0.45 * len(s) + 1.5))
    sns.heatmap(s, annot=True, fmt=".0f", cmap="Reds", vmin=0, vmax=100, cbar_kws={"label": "recall drop %"})
    plt.title("Relative recall drop under attack (%)"); plt.tight_layout(); plt.show()

**Takeaways to write in the thesis**
* Report robustness as **drop relative to clean performance**, otherwise a model that is bad on clean data looks "robust" because it has nothing to lose.
* A GNN advantage is meaningful when it holds across *both* domains and *all* attack directions; mention explicitly any attack where a baseline is more robust.
* Limitation: feature-space attacks ignore functionality constraints, and the PE attack uses the same GAN family used for augmentation, so the hardened model may be advantaged on this specific perturbation.